# Instrucciones de uso

In [ ]:
# A MASTER.txt file should be uploaded and populated with certification, TRF or system data where appropriate in the following order:

# none trf
# none sist
# none cert
# certificadora  (script)
# fin cert
# sistema        (script)
# fin sist
# TRF            (script)

# Selección modoe de uso:

In [ ]:
# search_in_cert(lines_cert)
# search_in_trf(lines_trf)
# search_sist(lines_sist)
comparacion(lines_cert, lines_sist, lines_trf) 

In [1]:
import re
import pandas as pd
pd.options.display.max_colwidth = 100
master_dict = {'Informe':'?', 'Cliente':'?', 'Dirección':'?',  'Equipo':'?', 'Marca':'?', 'Modelo':'?', 'Certificadora':'?',
                'Solicitud cert':'?', 'ID cert':'?', 'Norma':'?', 'Pag idem informe':'?', 'Palabra error':'?', 'Letra F':'?',
                'Fecha de Ingreso':'?','Venc. Cert.':'?'}
list_rows_with_F = []


# Separación de MASTER.txt

In [ ]:
# Estas funciones separan del MASTER.txt las secciones de certificadora, sistema y TRF.

with open(r'MASTER.txt', 'r', encoding='utf8')  as fp:
    lines = fp.readlines()

def search_body_cert(lines):  
  dict_1 = master_dict.copy()
  for row in lines[:5]:
    if re.search(r'none cert', row):
      for key in dict_1:
        dict_1[key] = '-'
      return dict_1
  else:
    for indice, row in enumerate(lines):
      if re.search(r'fin cert', row):
        return lines[:indice]

def search_body_sist(lines):
  dict_1 = master_dict.copy()
  for row in lines[:5]:
    if re.search(r'none sist', row):
      for key in dict_1:
        dict_1[key] = '-'
      return dict_1
  else:
    for indice, row in enumerate(lines):
      if re.search(r'fin cert', row):
        var = indice + 1
        for indice2, row2 in enumerate(lines[var:]):
          if re.search(r'fin sist', row2):
            return lines[var: var + indice2]

def search_body_trf(lines):
  dict_1 = master_dict.copy()
  for row in lines[:5]:
    if re.search(r'none trf', row):
      for key in dict_1:
        dict_1[key] = '-'
      return dict_1
  else:
    for indice, row in enumerate(lines):
      if re.search(r'fin sist', row):
        return lines[indice+1:]

In [4]:
lines_cert = search_body_cert(lines)
lines_sist = search_body_sist(lines)
lines_trf = search_body_trf(lines)

# Búsqueda de certificadora

In [ ]:
# Busca a que certificadora corresponde el .txt subido y usa la función correspondiente a dicha certificadora para la extracción de los datos a comparar.

def search_in_cert(lines_cert):
  dict_cert2 = {}
  if type(lines_cert) == dict:
    dict_cert2 = lines_cert

  for row in lines_cert:
    if re.search(r'Intertek', row, re.IGNORECASE):
      dict_cert2 =  search_intertek(lines_cert)

    elif re.search(r'qetkra', row, re.IGNORECASE):
      dict_cert2 = search_qetkra(lines_cert)

    elif re.search(r'TÜV', row, re.IGNORECASE):
      dict_cert2 = search_tuv(lines_cert)

    elif re.search(r'Net Connection International', row, re.IGNORECASE):
      dict_cert2 = search_ncc(lines_cert)

    elif re.search(r'Acta de extracción de muestras', row, re.IGNORECASE):
      dict_cert2 = search_lenor(lines_cert)

    elif re.search(r'Acta de Identificación de Muestras', row, re.IGNORECASE):
      dict_cert2 = search_iram(lines_cert)

    elif re.search(r'EJEMPLAR PARA EL LABORATORIO', row, re.IGNORECASE):
      dict_cert2 = search_bve(lines_cert)

    dict_cert2['Informe'] = '-'
    dict_cert2['Dirección'] = '-'
    dict_cert2['Pag idem informe'] = '-'
    dict_cert2['Palabra error'] = '-'
    dict_cert2['Fecha de Ingreso'] = '-'
    dict_cert2['Venc. Cert.'] = '-'
    dict_cert2['Letra F'] = '-'

  return dict_cert2

## Qetkra

In [6]:
def search_qetkra(lines_cert):
  dict_qetkra = master_dict.copy()
  dict_qetkra['Certificadora'] = 'Qetkra'
  for row in lines_cert:
    if re.search(r'Razón Social', row):
      dict_qetkra['Cliente'] = (' ').join(row.split(' ')[2:]).rstrip().lstrip()
    if re.search(r'Descripción', row):
      dict_qetkra['Equipo'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Marca', row):
      dict_qetkra['Marca'] = row.split(' ')[1]
    if re.search(r'Modelo a Ensayar', row):
      dict_qetkra['Modelo'] = (' ').join(row.split(' ')[3:]).rstrip().lstrip()
    if re.search(r'# de Certificado / Proceso', row):
      dict_qetkra['Solicitud cert'] = (' ').join(row.split(' ')[5:]).rstrip().lstrip()
    if re.search(r'# de Lacre', row):
      dict_qetkra['ID cert'] = (' ').join(row.split(' ')[3:]).rstrip().lstrip()
    if re.search(r'Norma de Aplicación', row):
      dict_qetkra['Norma'] = (' ').join(row.split(' ')[3:]).rstrip().lstrip()
  return dict_qetkra

## IRAM

In [7]:
def search_iram(lines_cert):

  dict_iram = master_dict.copy()
  dict_iram['Certificadora'] = 'IRAM'
  TF_modelo = True
  for index, row in enumerate(lines_cert):
    if re.search(r'Solicitante:', row):
      dict_iram['Cliente'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Producto', row):
      dict_iram['Equipo'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Modelo:', row) and TF_modelo == True:
      var_1 = row.split(':')[1].rstrip().lstrip()
      dict_iram['Modelo'] = var_1.split(' ')[0].rstrip().lstrip()
      TF_modelo = False
    if re.search(r'Datos de los especímenes', row):
      dict_iram['Marca'] = lines_cert[index+1].split(' ')[1].rstrip().lstrip()
    if re.search(r'normativo/s', row):
      dict_iram['Norma'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'DC ', row):
      var_1 = row.split('DC ')[1].rstrip().lstrip()
      dict_iram['Solicitud cert'] = var_1.split(' ')[0].rstrip().lstrip()
    if re.search(r'ETIQUETA', row):
      dict_iram['ID cert'] = row[:29].rstrip().lstrip()
  return dict_iram

## NCC

In [8]:
def search_ncc(lines_cert):

  dict_1 = {'Cliente':'', 'Equipo':'', 'Marca':'', 'Modelo':'', 'Certificadora':'NCC',
            'Solicitud cert':'', 'ID cert':'', 'Norma':'' }

  list_1 = []

  regex_solicitante = re.compile('(Solicitante)')
  regex_producto = re.compile('(Producto)')
  regex_marca = re.compile('(Marca)')
  regex_modelo = re.compile('(Modelo)')
  regex_solicitud = re.compile('(Nº de Certificado)')
  regex_id = re.compile('(Nº Identificación muestra )')
  regex_id_norm = re.compile('(Norma)')

  for row in lines_cert:

    if re.match(regex_solicitante, row)!= None:
      dict_1['Cliente'] = ' '.join(row.split()[1:])

    if re.match(regex_producto, row)!= None:
      dict_1['Equipo'] = ' '.join(row.split()[1:])

    if re.match(regex_marca, row)!= None:
      dict_1['Marca'] = ' '.join(row.split()[1:])

    if re.match(regex_modelo, row)!= None:
      dict_1['Modelo'] = ' '.join(row.split()[3:])

    if re.match(regex_solicitud, row)!= None:
      dict_1['Solicitud cert'] = ' '.join(row.split()[3:])

    if re.match(regex_id, row)!= None:
      dict_1['ID cert'] = ' '.join(row.split()[3:])

    if re.match(regex_id_norm, row)!= None:
      dict_1['Norma'] = ' '.join(row.split()[1:])

  return  dict_1

## TUV

In [9]:
def search_tuv(lines_cert):
  dict_tuv = master_dict.copy()
  dict_tuv['Certificadora'] = 'TÜV Rheinland'
  for row in lines_cert:
    if re.search(r'Cliente', row):
      dict_tuv['Cliente'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Producto:', row):
      dict_tuv['Equipo'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Modelo/s:', row):
      dict_tuv['Modelo'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Marca comercial:', row):
      dict_tuv['Marca'] = (' ').join(row.split(' ')[2:]).rstrip().lstrip()
    if re.search(r'Norma a utilizar:', row):
      dict_tuv['Norma'] = (' ').join(row.split(' ')[3:]).rstrip().lstrip()
    if re.search(r'Número de referencia', row):
      dict_tuv['Solicitud cert'] = (' ').join(row.split(' ')[3:]).rstrip().lstrip()
      dict_tuv['ID cert'] = (' ').join(row.split(' ')[3:]).rstrip().lstrip()
  return dict_tuv

## LENOR OCP

In [10]:
def search_lenor(lines_cert):
  dict_lenor = master_dict.copy()
  dict_lenor['Certificadora'] = 'Lenor OCP'
  for index, row in enumerate(lines_cert):
    if re.search(r'LCS-', row):
      dict_lenor['Cliente'] = lines_cert[index+1].rstrip().lstrip()
      dict_lenor['Norma'] = lines_cert[index+6].rstrip().lstrip()
      dict_lenor['Solicitud cert'] = lines_cert[index].rstrip().lstrip()
    if re.search(r'Fecha de Emisión', row):
      dict_lenor['Equipo'] = lines_cert[index+1].rstrip().lstrip()
    if re.search(r'Etiqueta N° Cantidad Marca Modelo', row):
      idCert_marca_modelo = lines_cert[index+1].split('Entrada')[0].rstrip().lstrip()
      dict_lenor['ID cert'] = idCert_marca_modelo.split(' ')[0].rstrip().lstrip()
      dict_lenor['Marca'] = (' ').join(idCert_marca_modelo.split(' ')[2:-1])
      dict_lenor['Modelo'] = idCert_marca_modelo.split(' ')[-1].rstrip().lstrip()
  return dict_lenor

## BVE

In [11]:
def search_bve(lines_cert):
  dict_bve = master_dict.copy()
  dict_bve['Certificadora'] = 'BVE'
  TF_marca = True
  for index, row in enumerate(lines_cert):
    if re.search(r'Cliente:', row):
      dict_bve['Cliente'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Producto:', row):
      dict_bve['Equipo'] = (' ').join(row.split(' ')[1:]).rstrip().lstrip()
    if re.search(r'Marca:', row) and  TF_marca == True:
      dict_bve['Marca'] = lines_cert[index+3].split(' ')[0].rstrip().lstrip()
      dict_bve['Modelo'] = lines_cert[index+6].rstrip().lstrip()
      dict_bve['Solicitud cert'] = lines_cert[index+2].rstrip().lstrip()
      dict_bve['Norma'] = (' ').join(lines_cert[index+3].split('Completo')[1:]).rstrip().lstrip()
      TF_marca = False
    if re.search(r'Familia:', row):
      dict_bve['ID cert'] = row.split(' ')[-1].rstrip().lstrip()
  return dict_bve

## Intertek

In [12]:
def search_intertek(lines_cert):
  cliente_tf = True
  dict_cert = master_dict.copy()
  dict_cert['Certificadora'] = 'Intertek'
  for index, row in enumerate(lines_cert):
    if re.search('CLIENTE', row) and cliente_tf == True:
        dict_cert['Cliente'] = row.split(':')[1].rstrip().lstrip()
        cliente_tf = False
    if re.search('DECLARADO', row):
        dict_cert['Marca'] = row.split(' ')[1].rstrip().lstrip()
    if re.search('DECLARADO', row):
        dict_cert['Modelo'] = row.split(' ')[2].rstrip().lstrip()
    if re.search(r'DESCRIPCIÓN DEL PRODUCTO', row):
        dict_cert['Equipo'] = lines_cert[index+1].split('.')[0].rstrip().lstrip()
    if re.search(r'CERTIFICADO:', row):
        dict_cert['Solicitud cert'] = row.split(':')[1].rstrip().lstrip()
    if re.search(r'ITK', row):
        dict_cert['ID cert'] = row.split(' ')[-1].rstrip().lstrip()
    if re.search(r'en esta solicitud', row):
        dict_cert['Norma'] = lines_cert[index+1].rstrip().lstrip()
  return dict_cert

# Búsqueda de TRFs

In [ ]:
# Busca a que TRF corresponde el .txt subido y usa la función correspondiente a dicho TRF para la extracción de los datos a comparar.

def search_in_trf(lines_trf):

  if type(lines_trf) == dict:
    dict_trf = lines_trf
    return dict_trf
  lista=[]


  for indice, row in enumerate(lines_trf):
    if re.search('62368-1', row):
      dict_final =  search_in_trf_368(lines_trf)

    elif re.search('60950-1', row):
      dict_final = search_in_trf_950(lines_trf)

    elif re.search('60065', row):
      dict_final = search_in_trf_065(lines_trf)

    elif re.search('63074', row):
      dict_final = search_in_trf_074(lines_trf)

    elif re.search('61558', row):
      dict_final = search_in_trf_558(lines_trf)

  dict_final['Fecha de Ingreso'] = '-'
  dict_final['Venc. Cert.'] = '-'

  return dict_final

## 61558

In [15]:
def search_in_trf_558(lines_trf):

  dict_1 = master_dict.copy()
  new_lines_trf = lines_trf[:250]
  tf_direccion = True
  count_id_cert = 0
  for index, row in enumerate(new_lines_trf):
    if re.search(r'Informe No.:', row):
      dict_1['Informe'] = row.split(' ')[-1].rstrip().lstrip()
    if re.search(r'Solicitante ', row):
      dict_1['Cliente'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Descripción del ítem bajo ensayo', row):
      dict_1['Equipo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitante ....', row) and tf_direccion == True:
      dict_1['Dirección'] = (' ').join(new_lines_trf[index+1].split(':')[1:]).rstrip().lstrip()
      tf_direccion = False
    if re.search(r'Marca registrada ...', row):
      dict_1['Marca'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Modelo/Tipo de referencia ....', row):
      dict_1['Modelo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Certificadora', row):
      dict_1['Certificadora'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitud', row):
      dict_1['Solicitud cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Identificación.....', row) and count_id_cert<2:
      dict_1['ID cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
      count_id_cert+=1
    if re.search(r'Norma', row):
      dict_1['Norma'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()

  return dict_1

## 62368

In [16]:
def search_in_trf_368(lines_trf):

  dict_1 = master_dict.copy()
  new_lines_trf = lines_trf[:140]
  for row in new_lines_trf:
    if re.search(r'Informe No.:', row):
      dict_1['Informe'] = row.split(' ')[-1].rstrip().lstrip()
    if re.search(r'Nombre del Solicitante', row):
      dict_1['Cliente'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Descripción del ítem de ensayo', row):
      dict_1['Equipo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Dirección', row):
      dict_1['Dirección'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Marca Registrada', row):
      dict_1['Marca'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Modelo y/o referencia tipo', row):
      dict_1['Modelo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Certificadora', row):
      dict_1['Certificadora'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitud', row):
      dict_1['Solicitud cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Identificación', row):
      dict_1['ID cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Norma', row):
      dict_1['Norma'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()

  return dict_1

## 60065

In [17]:
def search_in_trf_065(lines_trf):

  dict_1 = master_dict.copy()
  new_lines_trf = lines_trf[:140]
  for row in new_lines_trf:
    if re.search(r'Informe No.:', row):
      dict_1['Informe'] = row.split(' ')[-1].rstrip().lstrip()
    if re.search(r'Nombre del Cliente', row):
      dict_1['Cliente'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Descripción del ítem de ensayo', row):
      dict_1['Equipo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Domicilio', row):
      dict_1['Dirección'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Marca', row):
      dict_1['Marca'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Modelo y/o referencia de tipo', row):
      dict_1['Modelo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Certificadora', row):
      dict_1['Certificadora'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitud', row):
      dict_1['Solicitud cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Identificación', row):
      dict_1['ID cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Norma', row):
      dict_1['Norma'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()

  return dict_1

## 60950

In [18]:
def search_in_trf_950(lines_trf):
  dict_1 = master_dict.copy()
  new_lines_trf = lines_trf[:100]
  for row in new_lines_trf:
    if re.search(r'Informe No.:', row):
      dict_1['Informe'] = row.split(' ')[-1].rstrip().lstrip()
    if re.search(r'Nombre del Solicitante', row):
      dict_1['Cliente'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Descripción del ítem de ensayo', row):
      dict_1['Equipo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Dirección', row):
      dict_1['Dirección'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Marca Registrada', row):
      dict_1['Marca'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Modelo y/o referencia tipo', row):
      dict_1['Modelo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Certificadora', row):
      dict_1['Certificadora'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitud', row):
      dict_1['Solicitud cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Identificación', row):
      dict_1['ID cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Norma', row):
      dict_1['Norma'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()

  return dict_1

## 074

In [19]:
def search_in_trf_074(lines_trf):
  dict_1 = master_dict.copy()
  new_lines_trf = lines_trf[:130]
  for index, row in enumerate(new_lines_trf):
    if re.search(r'INFORME Nº:', row):
      dict_1['Informe'] = row.split(' ')[-1].rstrip().lstrip()
    if re.search(r'Solicitante', row):
      dict_1['Cliente'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Descripción del ítem ensayado', row):
      dict_1['Equipo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitante', row):
      dict_1['Dirección'] = (' ').join(new_lines_trf[index+1].split(':')[1:]).rstrip().lstrip()
    if re.search(r'Marca Registrada', row):
      dict_1['Marca'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Referencia Modelo/Tipo', row):
      dict_1['Modelo'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Certificadora', row):
      dict_1['Certificadora'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Solicitud', row):
      dict_1['Solicitud cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Identificación', row):
      dict_1['ID cert'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
    if re.search(r'Norma', row):
      dict_1['Norma'] = (' ').join(row.split(':')[1:]).rstrip().lstrip()
  return dict_1

# Búsqueda en sistema

In [ ]:
# Extae la información correspondiente a la parte del sistema en el .txt

def search_sist(lines):

  if type(lines) == dict:
    return lines

  dict_sist = master_dict.copy()
  dict_sist['Certificadora'] = '-'
  dict_sist['Solicitud cert'] = '-'
  dict_sist['ID cert'] = '-'
  dict_sist['Pag idem informe'] = '-'
  dict_sist['Palabra error'] = '-'
  dict_sist['Letra F'] = '-'

  for index, row in enumerate(lines):
    if re.search(r'Fecha de alta', row):
      remove_tabs = (' ').join(lines[index+2].split('\t')).rstrip().lstrip()
      dict_sist['Cliente'] = (' ').join(remove_tabs.split(' ')[1:]).rstrip().lstrip()
        # cliente_tf = False

    if re.search('Descripción del ensayo', row):
      try:
        dict_sist['Equipo'] = lines[index+6].split('/')[1].rstrip().lstrip()
        dict_sist['Marca'] = lines[index+6].split('/')[2].rstrip().lstrip()
        dict_sist['Modelo'] = lines[index+6].split('/')[3].rstrip().lstrip()

      except:
        dict_sist['Equipo'] = lines[index+5].split('/')[1].rstrip().lstrip()
        dict_sist['Marca'] = lines[index+5].split('/')[2].rstrip().lstrip()
        dict_sist['Modelo'] = lines[index+5].split('/')[3].rstrip().lstrip()

    try:
      if re.search('Fecha de ingreso', row):
        dict_sist['Fecha de Ingreso'] = lines[index+1].split('\t')[1].rstrip().lstrip()

    except:
      if re.search('Fecha de ingreso', row):
        dict_sist['Fecha de Ingreso'] = lines[index+2].split('\t')[1].rstrip().lstrip()


    if re.search('Set / # Informe', row):
      dict_sist['Informe'] = lines[index+1].split('/')[1][:-3].rstrip().lstrip()
      dict_sist['Dirección'] = lines[index+4].rstrip().lstrip()

    if re.search('Agregar Normas', row):
      dict_sist['Norma'] = ('-').join(lines[index+3].split('-')[1:]).rstrip().lstrip()

    if re.search('Venc. Cert.', row):
      dict_sist['Venc. Cert.'] = lines[index+1].rstrip().lstrip()


  return dict_sist

# Códigos varios

In [21]:
def switch(post_o, post_f, list_var):
  a = list_var[post_o]
  b = list_var[post_f]
  list_var[post_o] = b
  list_var[post_f] = a

  return list_var

def count_num_trf(lines_trf): # Cuenta la cantidad de numero
  if type(lines_trf) == dict:
    return '-'
  lines_num = lines_trf.copy()
  pag_contadas=0
  num_inf = lines_num[0].split()[-1]
  pag_tot = lines_num[0].split()[3]
  lines_num.pop(0)
  for row in lines_num:
    for a in row.split():
      if a == num_inf:
        pag_contadas+=1

  return str(pag_contadas) + '/' + str(pag_tot)

def buscador_error(lines_trf):
  if type(lines_trf) == dict:
    return '-'
  lines_error = lines_trf.copy()
  errores=0
  for row in lines_error:
    if re.search(re.compile(r'error', re.IGNORECASE), row)!= None:
      errores += 1

  return errores

def buscador_letra_f(lines_trf):
  if type(lines_trf) == dict:
    return '-'
  lines_efes = lines_trf.copy()
  efes=0
  for index, row in enumerate(lines_efes):
    if re.search(r'f$', row, re.IGNORECASE):
      efes += 1
      print(lines_efes[index-2], lines_efes[index-1], lines_efes[index])
    if re.search(r' f ', row, re.IGNORECASE):
      efes += 1
      print(lines_efes[index-2], lines_efes[index-1], lines_efes[index])

  return efes

def comparacion(lines_cert, lines_sist, lines_trf):
  dict_cert = search_in_cert(lines_cert)
  dict_sist = search_sist(lines_sist)
  dict_trf = search_in_trf(lines_trf)
  dict_comp = master_dict.copy()

    # for key, value in dict_comp.items():
  #   dict_comp[key] = [dict_trf[key], dict_cert[key], dict_trf[key]== dict_cert[key]]

  df_rows = pd.DataFrame({'--':['TRF', 'CERT', 'SIST']})
  dict_trf['Pag idem informe'] = count_num_trf(lines_trf)
  dict_trf['Palabra error'] = buscador_error(lines_trf)
  dict_trf['Letra F'] = buscador_letra_f(lines_trf)
  # df_comparativo = pd.DataFrame(dict_comp)


  todo = pd.DataFrame([dict_trf, dict_cert,dict_sist])

  return pd.concat([df_rows,todo], axis=1 ).set_index('--')